# Librerias


In [14]:
# cargo las librerias que necesito
require("data.table")
require("lightgbm")
library(data.table)


# URLS

Ejecutar con el directorio de trabajo en `entregas/`.


In [15]:
carpeta_entregas <- getwd()
data_folder <- file.path(carpeta_entregas, "dataset")
carpeta_parametros <- file.path(carpeta_entregas, "parametros")
carpeta_salida <- file.path(carpeta_entregas, "entregas")
archivo_logging <- file.path(carpeta_entregas, "logging.csv")


# Parametrizacion de la prueba

Completar el mes a predecir, el ID del `.rds` y la cantidad de envios antes de ejecutar. Revisar los meses de entrenamiento.


In [16]:
files = list.files(data_folder)
cat(files, sep = "\n")

c_01_tern_2_lags_O2.csv
c_01_tern_2_lags.csv
c_01_tern_2_lf_k6_p4.csv
c_01_tern_2_lf_O2_k3_p6.csv
c_01_tern_2.csv
competencia_01_crudo.csv
competencia_01_ternaria_lags.csv
competencia_01_ternaria_lf_k6_perdida4.csv
competencia_01_ternaria.csv
versiones_dataset_1.ipynb
versiones_dataset_2.ipynb


In [17]:
# dataset_file <- "competencia_01_ternaria_lags.csv"
# dataset_file <- "competencia_01_ternaria_lf_k6_perdida4.csv"
dataset_file <- "c_01_tern_2_lf_k6_p4.csv"

train_final <- c(202103, 202104,202105, 202106)
future <- 202108  # mes a predecir
SEEDS <- c(230047, 230059, 230063, 230077, 230081)

id_rds <- 44  # archivo parametros/<id_rds>.rds
envios <- 11000 # cantidad de clientes con Predicted = 1


In [18]:

columnas_excluir <- c(
    "ternaria",
    'ternaria_lag1',
    "clase01",
    "fold",
    "azar",
    "training",
    'cprestamos_personales',
    'mprestamos_personales',
    'cprestamos_personales_lag1',
    'mprestamos_personales_lag1',
    'cprestamos_personales_delta1',
    'mprestamos_personales_delta1',
    'cprestamos_personales_lag_pond',
    'mprestamos_personales_lag_pond',
    'cprestamos_personales_delta_lag_pond',
    'mprestamos_personales_delta_lag_pond'
)

if (anyNA(c(future, id_rds, envios))) {
  stop("Completar future, id_rds y envios en esta celda.")
}


# LEEMOS DATASET


In [19]:
dataset_url <- file.path(data_folder, dataset_file)
dataset <- fread(dataset_url)

setorder(dataset, foto_mes, ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, ternaria)]


foto_mes,ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,continua,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,continua,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,continua,161755


# Parametros

El `.rds` debe estar copiado en `parametros/`. Se usan los parametros guardados, sin volver a ajustar `min_data_in_leaf`.


In [20]:
parametros <- readRDS(
  file.path(carpeta_parametros, paste0(id_rds, ".rds"))
)


# Dataset train y prediccion


In [21]:
dataset[, clase01 := ifelse(ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

campos_buenos <- setdiff(colnames(dataset), columnas_excluir)

dataset_train <- dataset[foto_mes %in% train_final]
dfuture <- dataset[foto_mes %in% future]

if (length(future) != 1L || nrow(dataset_train) == 0L ||
    nrow(dfuture) == 0L || envios < 1L || envios > nrow(dfuture) ||
    envios != as.integer(envios)) {
  stop("Revisar meses y envios: se requiere un mes a predecir y filas en ambos datasets.")
}

dataset_train[, .N, list(foto_mes, ternaria)]


foto_mes,ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,continua,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,continua,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,continua,161755


# Entreno y genero salidas

Se conserva solamente el CSV actual de cada semilla.


In [22]:
if (file.exists(archivo_logging)) {
  historico <- fread(archivo_logging)
  id_prueba <- max(historico$id_prueba) + 1L
} else {
  id_prueba <- 1L
}

dir.create(carpeta_salida, recursive= TRUE, showWarnings= FALSE)
archivos_salida <- character(length(SEEDS))

for (i in seq_along(SEEDS)) {
  # El .rds puede contener parametros individuales o una lista por semilla.
  if ("num_iterations" %in% names(parametros)) {
    params_semilla <- parametros
  } else {
    params_semilla <- parametros[[as.character(SEEDS[i])]]
    if (is.null(params_semilla)) {
      stop("El .rds no contiene parametros para la semilla ", SEEDS[i])
    }
  }
  params_semilla$seed <- SEEDS[i]

  dtrain_final <- lgb.Dataset(
    data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
    label= dataset_train[, clase01]
  )

  modelo_final <- lgb.train(
    data= dtrain_final,
    param= params_semilla
  )

  prediccion <- predict(
    modelo_final,
    data.matrix(dfuture[, campos_buenos, with= FALSE])
  )

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]
  setorder(tb_prediccion, -prob)
  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  # archivos_salida[i] <- paste0("r001_", SEEDS[i], ".csv")
  archivos_salida[i] <- paste0("r001_", SEEDS[i],"_", id_prueba, ".csv")

  fwrite(
    tb_prediccion[Predicted == 1L, list(numero_de_cliente)],
    file.path(carpeta_salida, archivos_salida[i]),
    sep= ",",
    col.names= FALSE
  )

  cat("Semilla=", SEEDS[i], " Archivo=", archivos_salida[i], "\n", sep= "")
}

Semilla=230047 Archivo=r001_230047_13.csv
Semilla=230059 Archivo=r001_230059_13.csv
Semilla=230063 Archivo=r001_230063_13.csv
Semilla=230077 Archivo=r001_230077_13.csv
Semilla=230081 Archivo=r001_230081_13.csv


# Documentacion de la prueba

Una fila por ejecucion, con todas las semillas usadas. Completar la nota y el resultado de Kaggle como texto. Cuando llegue el resultado, ejecutar solamente esta celda: actualiza la misma fila y conserva las pruebas anteriores.


In [23]:
#lets stop here
pepe

ERROR: Error in eval(expr, envir, enclos): object 'pepe' not found


In [24]:
nota <- "agregamos otro mes al entrenamiento dataset fraccionario con nan en ccajas_depositos para 202105"
resultado_competencia_mean <- "102.9930"  # score de Kaggle o referencia a un problema con la entrega
resultado_competencia_std <- "4.0862"
ref_submit <- 'e13_rbis'

In [25]:

registro_prueba <- data.table(
  id_prueba = id_prueba,
  ref_submit = ref_submit,
  notebook = "r001.ipynb",
  dataset = dataset_file,
  seeds = paste(SEEDS, collapse = ","),
  cantidad_registros_totales = nrow(dataset),
  cantidad_registros_entrenamiento = nrow(dataset_train),
  cantidad_registros_prediccion = nrow(dfuture),
  meses_entrenamiento = paste(train_final, collapse = ","),
  meses_prediccion = paste(future, collapse = ","),
  columnas_excluidas = paste(columnas_excluir, collapse = ","),
  "bajas" = "1+2",                                       #baja+1 y baja+2 o solo baja+2
  id_rds = id_rds,
  envios = envios,
  archivos = paste(archivos_salida, collapse = ","),
  nota = nota,
  resultado_competencia_mean = resultado_competencia_mean,
  resultado_competencia_std = resultado_competencia_std
)

if (file.exists(archivo_logging)) {
  historico <- fread(
    archivo_logging,
    colClasses = list(character = c("nota", "resultado_competencia_mean"))
  )
  fila <- which(historico$id_prueba == id_prueba)

  if (length(fila) > 0L) {
    historico[fila, (names(registro_prueba)) := registro_prueba]
    fwrite(historico, archivo_logging)
  } else {
    fwrite(registro_prueba, archivo_logging, append = TRUE, col.names = FALSE)
  }
} else {
  fwrite(registro_prueba, archivo_logging)
}


In [26]:

# registro_prueba
tail(fread(archivo_logging), 10)


id_prueba,ref_submit,notebook,dataset,seeds,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_prediccion,meses_entrenamiento,meses_prediccion,columnas_excluidas,bajas,id_rds,envios,archivos,nota,resultado_competencia_mean,resultado_competencia_std
<int>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>
4,e4_r,r001.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,training,cprestamos_personales,mprestamos_personales,cprestamos_personales_lag1,mprestamos_personales_lag1,cprestamos_personales_delta1,mprestamos_personales_delta1,cprestamos_personales_lag_pond,mprestamos_personales_lag_pond,cprestamos_personales_delta_lag_pond,mprestamos_personales_delta_lag_pond",1+2,44,11500,"r001_230047_4.csv,r001_230059_4.csv,r001_230063_4.csv,r001_230077_4.csv,r001_230081_4.csv","cambiamos dataset al q se uso en rds 44, deltas fraccionarios",95.2215,1.7673
5,e5_r,r001.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,training,cprestamos_personales,mprestamos_personales,cprestamos_personales_lag1,mprestamos_personales_lag1,cprestamos_personales_delta1,mprestamos_personales_delta1,cprestamos_personales_lag_pond,mprestamos_personales_lag_pond,cprestamos_personales_delta_lag_pond,mprestamos_personales_delta_lag_pond",1+2,44,12000,"r001_230047_5.csv,r001_230059_5.csv,r001_230063_5.csv,r001_230077_5.csv,r001_230081_5.csv",e4_r pasando de 11.5k envios a 12k envios,95.3095,2.2131
6,e6_r,r001.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,training,cprestamos_personales,mprestamos_personales,cprestamos_personales_lag1,mprestamos_personales_lag1,cprestamos_personales_delta1,mprestamos_personales_delta1,cprestamos_personales_lag_pond,mprestamos_personales_lag_pond,cprestamos_personales_delta_lag_pond,mprestamos_personales_delta_lag_pond",1+2,44,11000,"r001_230047_6.csv,r001_230059_6.csv,r001_230063_6.csv,r001_230077_6.csv,r001_230081_6.csv",e4_r pasando de 11.5k envios a 11k envios,95.7825,1.9090
7,e7_r,r001.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,training,cprestamos_personales,mprestamos_personales,cprestamos_personales_lag1,mprestamos_personales_lag1,cprestamos_personales_delta1,mprestamos_personales_delta1,cprestamos_personales_lag_pond,mprestamos_personales_lag_pond,cprestamos_personales_delta_lag_pond,mprestamos_personales_delta_lag_pond",1+2,44,10500,"r001_230047_7.csv,r001_230059_7.csv,r001_230063_7.csv,r001_230077_7.csv,r001_230081_7.csv",e4_r pasando de 11.5k envios a 10.5k envios,95.3535,2.3168
8,e8_r,r002.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,818414,164647,"202103,202104,202105,202106,202107",202108,"ternaria,ternaria_lag1,clase01,fold,azar,training,cprestamos_personales,mprestamos_personales,cprestamos_personales_lag1,mprestamos_personales_lag1,cprestamos_personales_delta1,mprestamos_personales_delta1",1+2,53,10500,"r002_230047_8.csv,r002_230059_8.csv,r002_230063_8.csv,r002_230077_8.csv,r002_230081_8.csv","BO para predecir b+2 en 202107 y usada para entrenar para 202108 | BAJA+2 estimadas para 202107; id_rds_julio=52; meses_entrenamiento_julio=202103,202104,202105,202106; seed_julio=230047; prob_corte=0.0035446538114082874; BAJA+1 reales=1103; BAJA+2 estimadas=14888",91.3055,4.8356
9,e9_r,r002.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,818414,164647,"202103,202104,202105,202106,202107",202108,"ternaria,ternaria_lag1,clase01,fold,azar,training,cprestamos_personales,mpres